<a href="https://colab.research.google.com/github/emgrimm42/NLP-Hausarbeit/blob/Der-Freiheitskampf/ha_download__ber_api_der_ddb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download über API der DDB

Dieses Google Colab Notebook basiert auf den Forschungsdaten-Notebooks der Deutschen Nationalbibliothek, die unter folgendem Link verfügbar sind: https://github.com/Deutsche-Digitale-Bibliothek/ddblabs-summer-school-2024


## Suchindizes der Deutsche Digitale Bibliothek
Die Deutsche Digitale Bibliothek betreibt [Solr](https://solr.apache.org/guide/8_8/searching.html)-Suchindizes, die für die verschiedenen Funktionen der (Sub-) Portale benötigt werden. Das Zeitungsportal benutzt zwei Suchindizes. Eine weiterführende Dokumentation befindet sich hier: https://api.deutsche-digitale-bibliothek.de/#/search/getSolrSearch

- `newspaper`: enthält Informationen über Zeitungstitel
  - Schema: https://dev.fiz-karlsruhe.de/stash/projects/DDB/repos/ddb-backend/browse/Cortex/conf/solr/newspaper/conf/schema.xml
  - Konfiguration: https://dev.fiz-karlsruhe.de/stash/projects/DDB/repos/ddb-backend/browse/Cortex/conf/solr/newspaper/conf/solrconfig.xml
- `newspaper-issues`: enthält die zeitungsbezogenen Metadaten inkl. Volltexte
  - Schema: https://dev.fiz-karlsruhe.de/stash/projects/DDB/repos/ddb-backend/browse/Cortex/conf/solr/newspaper-issues/conf/schema.xml
  - Konfiguration: https://dev.fiz-karlsruhe.de/stash/projects/DDB/repos/ddb-backend/browse/Cortex/conf/solr/newspaper-issues/conf/solrconfig.xml

## Suchindex `newspaper`
Der Suchindex `newspaper` ist ein Suchindex über alle Zeitungstitel der [Zeitschriftendatenbank (ZDB)](https://zdb-katalog.de/). Im Schema des Suchindex (s.o.) sind die Suchfelder (Facetten) dokumentiert. Wenn Zeitungstitel gefunden werden sollen, die im Zeitungsportal verfügbar sind, dann muss nach `hasLoadedIssues:true` (Feld:Wert) gesucht werden.

- https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper/select?q=hasLoadedIssues:true

Die Suchfelder können beliebig miteinander kombiniert werden. Das geht mit den Operatoren `AND` und `OR`. Möchte man beispielsweise Zeitungen mit „La otra Alemania“ im Titel suchen, die auch im Zeitungsportal verfügbar sind, dann kombiniert man: `hasLoadedIssues:true AND title:"La otra Alemania"`

- [https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper/select?q=hasLoadedIssues:true AND title:"La otra Alemania"](https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper/select?q=hasLoadedIssues:true%20AND%20title:"La+otra+Alemania")

### Python-Programmcode

Python-Programmcode in Jupyter Notebooks ermöglicht interaktives Programmieren und sofortige Anzeige von Ergebnissen. Es ist ideal für die Datenanalyse und Visualisierungen. Python-Bibliotheken können in Notebooks nachgenutzt werden und erhöhen so den Funktionsumfang.

Die o. g. Solr-Abfrage kann mit Python ausgeführt werden (Bibliothek [`requests`](https://pypi.org/project/requests/)) und die Antwort von der API mit einem JSON-Parser (`json`) gelesen werden. Eine andere Möglichkeit ist, den Solr-Client [`pysolr`](https://pypi.org/project/pysolr/) zu benutzen. Dieser muss zunächst in der Umgebung mit `pip install -q pysolr` (oder ggf. mit Conda: `conda install conda-forge::pysolr`) installiert werden.

In [ ]:
# Python-Bibliothek pysolr installieren
%pip install -q pysolr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


### KI-generierter Programmcode

Die Erstellung des Python-Programmcodes ist KI-gestützt möglich. Die folgenden KI-Prompts sind mit ChatGPT (GPT-4o) erfolgreich getestet und liefern (oft 😉) das gewünschte Ergebnis.

<div class="alert alert-block alert-info">
<b>Prompt:</b> Ich möchte in Python mit dem Solr-Client <code>pysolr</code> auf den Endpunkt <code>https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper</code> zugreifen. Kannst Du mir einen Python-Code erstellen, der im Feld <code>location</code> nach „Buenos Aires“ sucht und auch <code>hasLoadedIssues</code> auf „wahr“ setzt. Gibt bitte <code>id</code>, <code>title</code>, <code>location</code>, <code>frequency</code> und <code>progress</code> für jeden Suchtreffer aus.
</div>

In [ ]:
import pysolr

# Solr-Endpunkt-URL
solr_url = 'https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper'

# Solr-Client initialisieren
solr = pysolr.Solr(solr_url, timeout=10)

# Suchparameter
q = {
    'q': 'location:"Dresden" AND hasLoadedIssues:true',
    'fl': 'id,title,location,frequency,progress',
    'rows': 100  # Anzahl der zurückzugebenden Ergebnisse (hier auf 100 gesetzt, anpassbar)
}

# Suche ausführen
results = solr.search(**q)

# Ergebnisse ausgeben
if results.hits > 0:
    print(f"Anzahl der Suchtreffer: {results.hits}\n")
    for result in results:
        print(f"ID: {result.get('id', 'N/A')}")
        print(f"Title: {result.get('title', 'N/A')}")
        print(f"Location: {result.get('location', 'N/A')}")
        print(f"Frequency: {result.get('frequency', 'N/A')}")
        print(f"Progress: {result.get('progress', 'N/A')}")
        print("---" * 10)
else:
    print("Keine Ergebnisse für die angegebene Suchanfrage gefunden.")

Anzahl der Suchtreffer: 63

ID: 2803953-1
Title: ['Der Freiheitskampf']
Location: ['Dresden']
Frequency: ['http://id.loc.gov/vocabulary/frequencies/dyl']
Progress: ['1930,1(1.Aug.) - 1932,50(29.Febr.)']
------------------------------
ID: 2924281-2
Title: ['Der Zeitgeist']
Location: ['Dresden']
Frequency: ['http://id.loc.gov/vocabulary/frequencies/mon']
Progress: ['Nr. 8.1920,19.Mai ; mehr nicht digitalisiert']
------------------------------
ID: 2904126-0
Title: ['Dresdner neueste Nachrichten']
Location: ['Dresden']
Frequency: ['http://id.loc.gov/vocabulary/frequencies/dyl']
Progress: ['11.1903,91(1.Apr.) - 51.1943,61(14.März); mehr nicht digitalisiert']
------------------------------
ID: 2889174-0
Title: ['Sächsische Staatszeitung', 'Sächsische Staatszeitung, Synodalbeilage']
Location: ['Dresden']
Frequency: ['http://id.loc.gov/vocabulary/frequencies/wkl']
Progress: ['1915,1-3; 1917,1-5; 1919,13-20; 1920/22,1 u.3-57; mehr nicht digitalisiert']
------------------------------
ID: 2846915

In [ ]:
import pysolr

# Solr-Endpunkt-URL
solr_url = 'https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper'

# Solr-Client initialisieren
solr = pysolr.Solr(solr_url, timeout=10)

# Suchparameter
q = {
    'q': 'title:"Der Freiheitskampf / A, [Dresden]" AND hasLoadedIssues:true', #Zeitungstitel im Zeitungsportal: https://www.deutsche-digitale-bibliothek.de/newspaper/select/title
    'fl': 'id,title,location,frequency,progress',
    'rows': 10  # Anzahl der zurückzugebenden Ergebnisse
}

# Suche ausführen
results = solr.search(**q)

# Ergebnisse ausgeben
for result in results:
    print(f"ID: {result.get('id', 'N/A')}")
    print(f"Title: {result.get('title', 'N/A')}")
    print(f"Location: {result.get('location', 'N/A')}")
    print(f"Frequency: {result.get('frequency', 'N/A')}")
    print(f"Progress: {result.get('progress', 'N/A')}")
    print("-" * 40)

ID: 2803974-9
Title: ['Der Freiheitskampf', 'Der Freiheitskampf / A, [Dresden]']
Location: ['Dresden']
Frequency: ['http://id.loc.gov/vocabulary/frequencies/dyl']
Progress: ['1935,2.Jan. - 1936,Apr.; 1936,20.Juli - 1937,Febr.; 1937,Mai - 1944,Dez.; 1945,37(13.Febr.)-105(8.Mai); mehr nicht digitalisiert']
----------------------------------------


## Suchindex `newspaper-issues`

Der Suchindex `newspaper-issues` ist ein weiterer Suchindex des Zeitungsportals. Dieser enthält alle Ausgaben (`type:issue`) einer Zeitung und alle Seiten (`type:page`). Wenn man nur in einem Zeitungstitel suchen möchte, so kann dies über `zdb_id:{ID der Zeitschriftendatenbank}` (für die „La otra Alemania“ ist es `zdb_id:2149754-0`) eingegrenzt werden.

### KI-generierter Programmcode

<div class="alert alert-block alert-info">
<b>Prompt:</b> Schreibe ein Python-Code, der mithilfe der <code>pysolr</code>-Bibliothek eine Suche in einem Solr-Index durchführt und die Ergebnisse in ein Pandas DataFrame überführt. Der Solr-Index ist über die URL <code>https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues</code> erreichbar. Die Suchabfrage soll nach Dokumenten mit der <code>zdb_id</code> „2149754-0“ und dem <code>type</code> „issue“ suchen und bis zu 1000 Ergebnisse zurückgeben. Anschließend sollen die Ergebnisse in ein Pandas DataFrame überführt und angezeigt werden.
</div>

In [ ]:
import pysolr
import pandas as pd

# Solr-Endpunkt-URL
solr_url = 'https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues'

# Solr-Client initialisieren
solr = pysolr.Solr(solr_url, timeout=10)

# Suchparameter
q = {
    'q': 'zdb_id:2803974-9 AND type:issue',
    'rows': 1000  # Anzahl der zurückzugebenden Ergebnisse (hier auf 1000 gesetzt)
}

# Suche ausführen
results = solr.search(**q)

# Überführen der Ergebnisse in ein Pandas DataFrame
df = pd.DataFrame(results.docs)

# DataFrame anzeigen
print(df)

                                   id  \
0    XCUYY3DRW3GRUOMLY53VDOZGP7DXJUKN   
1    R2L63LIJ6QN4SD6M2JNETCWA763IZTQJ   
2    YA3DKPBNWMXQ7NZ4K4KK3C3W54KPEZLH   
3    5WO4BQBJWFKIBB4WIWDEFBC6DWN4TGAY   
4    JDJEGBIX4IUH6BOVW7UXRCYG3ZFCJFXK   
..                                ...   
995  SCIS6BQ7XBZ57EL56AR7JO4VNYZ42RGS   
996  UTMYJHWHE6SJ4FDBU4LNNG6KDAMHEARN   
997  WS7P7ABRQGKZW4DTD2CUW2NMXP3ZCDHW   
998  VWNYLJERYFUIJN3LKBX5KQYI3RSS3I4R   
999  T5MH3PXCUZHZLY23UXPMN76Q2COBNNQC   

                                           paper_title  \
0    Der Freiheitskampf : amtliche Zeitung der NSDA...   
1    Der Freiheitskampf : amtliche Zeitung der NSDA...   
2    Der Freiheitskampf : amtliche Zeitung der NSDA...   
3    Der Freiheitskampf : amtliche Zeitung der NSDA...   
4    Der Freiheitskampf : amtliche Zeitung der NSDA...   
..                                                 ...   
995  Der Freiheitskampf : amtliche Zeitung der NSDA...   
996  Der Freiheitskampf : amtliche Zeitung 

**Von mir am 5.3.26 hinzugefügt**

Frage: Wie viele Digitale Issues der von mir ausgewählten version des Freiheitskampfes gibt es?

Promt den ich genutzt habe um ChatGPT meinen Code schreiben zu lassen:

Ich möchte ermittelt wie viele Ausgaben einer bestimmten Zeitung im deutschen Zeitungsarchiv digitalisiert sind. Schreibe ein Python-Code, der mithilfe der pysolr-Bibliothek eine Suche in einem Solr-Index durchführt und die Ergebnisse in ein Pandas DataFrame überführt. Der Solr-Index ist über die URL https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues erreichbar. Die Suchabfrage soll nach Dokumenten mit der zdb_id „2803974-9“ und dem type „issue“ suchen. Als Ergebnis hätte ich gerne die Zahl der Ausgaben als Integer.


In [ ]:
import pysolr
import pandas as pd

# Solr-Endpunkt
SOLR_URL = "https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues"

# Verbindung zum Solr-Index herstellen
solr = pysolr.Solr(SOLR_URL, always_commit=True, timeout=10)

# Suchquery
query = 'zdb_id:"2803974-9" AND type:"issue"'

# Anzahl der Treffer abfragen (rows=0 reicht, wenn nur numFound benötigt wird)
results = solr.search(query, rows=1000)

# Treffer in DataFrame überführen
docs = results.docs
df = pd.DataFrame(docs)

# Gesamtzahl der Ausgaben
num_issues = int(results.hits)

print("Anzahl der digitalisierten Ausgaben:", num_issues)

Anzahl der digitalisierten Ausgaben: 3511


In [ ]:
df.to_excel('results2.xlsx')

In [ ]:
import pandas as pd

solr_url = 'https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues'
solr = pysolr.Solr(solr_url, always_commit=True, timeout=100)

q = {
    'q': 'zdb_id:2803974-9 AND type:issue',
    'rows':1000
}

response = solr.search(**q)

# Überführen der Ergebnisse in ein Pandas DataFrame
df = pd.DataFrame(response.docs)

# DataFrame anzeigen
print(df)

                                   id  \
0    PRYJ6VFGZUEDMD7QE64FRZYQBDHPAEUP   
1    U6LOXACMZEAYCV7OQBLD3PBWBEZQQ2SB   
2    DV4CRYJHT46TMTQ6MESMEVGMNSJTDFTT   
3    Z33Y4X5C2JNJ7OBWOJDX2XDQ7NCTLLBM   
4    3E5SVT5MJ5OKEY2HWBYNA4RA2EFH6WAG   
..                                ...   
995  BMMJUFA3I2QW5M4NBKYGI2J5ULLJPDRT   
996  T26UJW75VQOJUOF5BPVAKQGLP6YJOQPC   
997  7Y5ZUMSBL7I3HBELW5FXEFVKS46P7KN5   
998  NWCVRXCRZ6VZL6BJQ2JU4EHH6BSLAR6Y   
999  P3I2IMVYLZV65R34JRT4S4HZ34I2D55K   

                                           paper_title  \
0    Der Freiheitskampf : amtliche Zeitung der NSDA...   
1    Der Freiheitskampf : amtliche Zeitung der NSDA...   
2    Der Freiheitskampf : amtliche Zeitung der NSDA...   
3    Der Freiheitskampf : amtliche Zeitung der NSDA...   
4    Der Freiheitskampf : amtliche Zeitung der NSDA...   
..                                                 ...   
995  Der Freiheitskampf : amtliche Zeitung der NSDA...   
996  Der Freiheitskampf : amtliche Zeitung 

In [ ]:
df.to_excel('results5.xlsx')

### Von mir am 9.3.26 hinzugefügt
---
Hatte probleme mit dem existierenden Code den Text aller digitalisierter Ausgaben zu erhalten, habe mir also von ChatGPT mit diesem Promt:

*Ich möchte mit den digitalisierten ausgaben einer Zeitung aus dem Deutschen Zeitungsarchiv arbeiten. Schreibe ein Python-Code, der mithilfe der pysolr-Bibliothek eine Suche in einem Solr-Index durchführt und die Ergebnisse in ein Pandas DataFrame überführt. Der Solr-Index ist über die URL https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues erreichbar. Die Suchabfrage soll nach Dokumenten mit der zdb_id „2803974-9“ und dem type „issue“ suchen. Als Ergebnis möchte ich den Datensatz als excel Dokument, die spalte mit dem tatsächlichen Text der Zeitungsaufgabe sollte plainpagefulltext heißen*

Passenden Code generieren lassen.

In [ ]:
import pysolr
import pandas as pd

# Solr-Endpunkt
solr_url = "https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues"

# Verbindung zu Solr herstellen
solr = pysolr.Solr(solr_url, always_commit=False, timeout=30)

# Suchquery
query = 'zdb_id:"2803974-9" AND type:"issue"'


# Suche ausführen
results = solr.search(
    query,
    **{
        "fl": ",".join(fields),
        "rows": 3512  # Anzahl der Dokumente pro Anfrage
    }
)

# Ergebnisse in Liste von Dictionaries umwandeln
docs = []
for doc in results:
    record = {}
    for field in fields:
        value = doc.get(field, None)
        # Falls Solr Listen zurückgibt → in String umwandeln
        if isinstance(value, list):
            value = " ".join(map(str, value))
        record[field] = value
    docs.append(record)

# DataFrame erstellen
df = pd.DataFrame(docs)

# Excel-Datei speichern
output_file = "zeitungsausgaben_zdb_2803974-9.2.xlsx"
df.to_excel(output_file, index=False)

print(f"Export abgeschlossen: {output_file}")

Export abgeschlossen: zeitungsausgaben_zdb_2803974-9.2.xlsx


**Problem:** Der Volltext scheint nicht in den runtergeladenen Datein zu sein. Habe also das ehste an einem funktionierenden Code (gibt den vollen Text aber nur Teilweise), was ich habe in ChaCPT eingefügt, in der Hoffnung das Chatty alle Einschränkungen rausstreicht.  

In [ ]:
import pysolr
import pandas as pd

solr_url = "https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues"

# Verbindung zum Solr-Index herstellen
solr = pysolr.Solr(solr_url, timeout=100)

# Suchparameter
params = {
    "q": 'zdb_id:"2803974-9" AND type:page',
    "rows": 3512,
    "fl": "id,title,date,zdb_id,type,plainpagefulltext"
}

# Suche ausführen
response = solr.search(**params)

# Ergebnisse in DataFrame überführen
df = pd.DataFrame(response.docs)

# Falls Felder als Liste kommen → in String umwandeln
for col in df.columns:
    df[col] = df[col].apply(lambda x: " ".join(x) if isinstance(x, list) else x)

# DataFrame anzeigen
print(df.head())

# Export nach Excel
output_file = "zeitungsseiten_2803974-9.xlsx"
df.to_excel(output_file, index=False)

print(f"Excel-Datei gespeichert: {output_file}")

                                                  id     zdb_id  \
0  FUUG6FBTWAL4PV4UZGOPPA4SVEXP4UTG-uuid-ba2b5c10...  2803974-9   
1  FUUG6FBTWAL4PV4UZGOPPA4SVEXP4UTG-uuid-abff7855...  2803974-9   
2  FUUG6FBTWAL4PV4UZGOPPA4SVEXP4UTG-uuid-cf982e88...  2803974-9   
3  FUUG6FBTWAL4PV4UZGOPPA4SVEXP4UTG-uuid-5de0a839...  2803974-9   
4  FUUG6FBTWAL4PV4UZGOPPA4SVEXP4UTG-uuid-ce000ff1...  2803974-9   

                                   plainpagefulltext  
0  1 - k - < i r -> '.i 4 1 Nr. Z0. Sonnabend, Z0...  
1  Är. 30. Seite 2 Sonnabend, 3Y. Januar 1937 M. ...  
2  M. 30 Seite 5 „Der Freihettskampf" Sonnabend, ...  
3  Nr. 3V. Seite 4 „Der FreiheNskampß" Sonnabend,...  
4  Der Fre 1 he? tskamps" t? Sonnabend, 30. Janua...  
Excel-Datei gespeichert: zeitungsseiten_2803974-9.xlsx


**Zwischenfazit:**
type:page scheint mir Volltext zu geben, aber nie mehr als 74 Ergebnisse. Während type:issue mir 3511 Ergebnisse liefert, aber ohne den Volltext.
issue steht für eine komplette Zeitungsnummer (Ausgabe) - enthält aber primär Metadaten und nicht den Volltext
page steht für eine einzelne Zeitungsseite innerhalb einer Ausgabe. - und enthält typischer weise den Volltext

Jetzt wäre also wichtig zu erfahren wieviele digitaliserte Seiten es von meiner Gewünschten Zeitung überhaupt gibt. Glücklicherweise habe ich dafür schon den fat perfekten code.

In [ ]:
import pysolr
import pandas as pd

# Solr-Endpunkt
SOLR_URL = "https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues"

# Verbindung zum Solr-Index herstellen
solr = pysolr.Solr(SOLR_URL, always_commit=True, timeout=10)

# Suchquery
query = 'zdb_id:"2803974-9" AND type:"page"'

# Anzahl der Treffer abfragen (rows=0 reicht, wenn nur numFound benötigt wird)
results = solr.search(query, rows=1000)

# Treffer in DataFrame überführen
docs = results.docs
df = pd.DataFrame(docs)

# Gesamtzahl der Ausgaben
num_issues = int(results.hits)

print("Anzahl der digitalisierten Seiten:", num_issues)

Anzahl der digitalisierten Seiten: 74


Es gibt tatsächlich also nur 74 digitalisierte Seiten. Das erscheint mir allerdings etwas wenig um darauf eine Analyse zu starten.

Eine mögliche Lösung wären die anderen in Dresden veröffentlichten Ausgaben des Freiheitskampfes. Jeweils zdb_id:2803953-1 und zdb_id:2803964-6.

Mal schauen wie viele digitaliserte diese Zeitungen jeweils haben.

In [ ]:
import pysolr
import pandas as pd

# Solr-Endpunkt
SOLR_URL = "https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues"

# Verbindung zum Solr-Index herstellen
solr = pysolr.Solr(SOLR_URL, always_commit=True, timeout=10)

# Suchquery
query = '(zdb_id:"2803953-1" AND type:"page") OR (zdb_id:"2803964-6" AND type:"page") OR (zdb_id:"2803974-9" AND type:"page")'

# Anzahl der Treffer abfragen (rows=0 reicht, wenn nur numFound benötigt wird)
results = solr.search(query, rows=1000)

# Treffer in DataFrame überführen
docs = results.docs
df = pd.DataFrame(docs)

# Gesamtzahl der Ausgaben
num_issues = int(results.hits)

print("Anzahl der digitalisierten Seiten:", num_issues)

Anzahl der digitalisierten Seiten: 106


2803953-1 hat keine digitalisierten Seiten online. 2803964-6 hat nur 32 Seiten online. Gibt es eventuell noch andere Freiheitskämpfe in anderen Städten?

Nope, gibt es nicht. Habe jetzt eine Mail an Dr. Oberbichler geschrieben, muss aber potenziell mein komplettes Thema ändern weil ich nicht genug Quellen habe.

### Erste Datenanalyse

In dem Dataframe können nun Datenanalysen vorgenommen werden. Wir wollen den Publikationszeitraum von der Zeitung „Der Freiheitskampf / A“ ermitteln.

In [ ]:
# Sicherstellen, dass publication_date als Datumswerte formatiert sind
df['publication_date'] = pd.to_datetime(df['publication_date'], errors="coerce")

# Frühestes und spätestes Datum ermitteln
earliest_date = df['publication_date'].min()
latest_date = df['publication_date'].max()

# Ergebnisse anzeigen
print(f"Frühestes Veröffentlichungsdatum: {earliest_date}")
print(f"Spätestes Veröffentlichungsdatum: {latest_date}")

Frühestes Veröffentlichungsdatum: 1935-12-31 12:00:00+00:00
Spätestes Veröffentlichungsdatum: 1937-01-30 12:00:00+00:00


Habe den obrigen Code durch eine addition von `errors="coerce"`in Zeile 2 nach `df['publication_date']`aber vor der schließenden Klammer korrigieren können. Vermutlich gab es Fehler beim Umwandeln zu datetime, die durch die coercion überwunden wurden statt dass sie wie ohne coercion einfach abgefallen sind.

Das oben genannte Späteste Veröffenltlichungsdatum erscheint mir nicht realistisch, da das deutsche Zeitungsportal behauptet es wurden exemplare bis 1945 digitalisiert, daher will ich dieselben Query noch mal mit eigenem code durchführen.

Hier der ChatGPT promt:
Ich möchte ermittelt wann die Ausgaben einer bestimmten Zeitung, die im deutschen Zeitungsarchiv digitalisiert sind, ursprünglich veröffentlicht wurden. Schreibe ein Python-Code, der mithilfe der pysolr-Bibliothek eine Suche in einem Solr-Index durchführt und die Ergebnisse in ein Pandas DataFrame überführt. Der Solr-Index ist über die URL https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues erreichbar. Die Suchabfrage soll nach Dokumenten mit der zdb_id „2803974-9“ und dem type „issue“ suchen. Als Ergebnis hätte ich gerne das Früheste Veröffentlichungsdatum und das Späteste Veröffentlichungsdatum.

Habe das ChatGPT ergebnis minimal abwandeln müssen, da chatty der meinung war das die datums-variable unter date gespeichert ist, tatsächlich ist sie aber unter publication_date gespeichert. Der Fix war alle Instanzen von date mit publication_date zu ersetzen.

In [ ]:
import pysolr
import pandas as pd

# Solr-Endpoint
solr_url = "https://api.deutsche-digitale-bibliothek.de/2/search/index/newspaper-issues"

# Verbindung herstellen
solr = pysolr.Solr(solr_url, always_commit=False, timeout=10)

# Suchquery
query = 'zdb_id:"2803974-9" AND type:"issue"'

# Felder, die wir abrufen möchten
fields = ["id", "publication_date"]

# Ergebnisse abrufen (ggf. rows erhöhen, falls sehr viele Treffer)
results = solr.search(query, fl=",".join(fields), rows=100000)

# Ergebnisse in Liste umwandeln
docs = [dict(doc) for doc in results]

# DataFrame erzeugen
df = pd.DataFrame(docs)

# Datum konvertieren
df["publication_date"] = pd.to_datetime(df['publication_date'], errors="coerce")

# Frühestes und spätestes Datum bestimmen
earliest_date = df["publication_date"].min()
latest_date = df["publication_date"].max()

print("Frühestes Veröffentlichungsdatum:", earliest_date)
print("Spätestes Veröffentlichungsdatum:", latest_date)

Frühestes Veröffentlichungsdatum: 1935-01-02 12:00:00+00:00
Spätestes Veröffentlichungsdatum: 1945-05-08 12:00:00+00:00
